# HabitIA · probar el estudio en Google Colab

Esta copia ejecuta las tablas, los gráficos y las comprobaciones del estudio de 2018 con el experimento guardado. **No inicia un entrenamiento nuevo.** Las salidas aparecen al ejecutar las celdas.

1. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, elige **CPU** y la versión **2026.07** (Python 3.12).
2. Pulsa **Ejecutar todo**. Cuando aparezca el selector, carga **HabitIA_Colab_datos_y_codigo.zip**, facilitado por el equipo. Contiene el código, los datos y los modelos de esta prueba.
3. Si la instalación pide reiniciar, selecciona **Entorno de ejecución → Reiniciar la sesión y ejecutar todas las celdas**. El ZIP permanece en la sesión y no tienes que volver a cargarlo.
4. Al final debe aparecer **PRUEBA COMPLETA**. Puedes modificar las celdas de análisis y ejecutarlas de nuevo.

El ZIP se carga en la máquina temporal de Colab de tu cuenta. No se publica ni se monta Google Drive. Cuando Colab elimina la máquina hay que cargarlo otra vez. Compartir el enlace al cuaderno no comparte estos archivos; quien lo ejecute necesita su propia copia del ZIP.

La primera instalación puede tardar varios minutos. No se necesita GPU, Idealista ni una API de pago. Estas métricas históricas no validan precios de compraventa ni precisión en 2026.


## A · Cargar y verificar los archivos

In [ ]:
from pathlib import Path, PurePosixPath
import sys, os, json, hashlib, tempfile, zipfile

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Selecciona CPU y versión 2026.07 en Entorno de ejecución → Cambiar tipo de entorno de ejecución (Python 3.12).')

ARCHIVE_SHA = '8d848432fc641363787ac127e50ff585f42da9b4d54c44cb117f5b1a9ca87bad'
archive_path = Path('/content/HabitIA_Colab_datos_y_codigo.zip')
if not archive_path.is_file() or hashlib.sha256(archive_path.read_bytes()).hexdigest() != ARCHIVE_SHA:
    from google.colab import files
    print('Selecciona HabitIA_Colab_datos_y_codigo.zip')
    uploaded = files.upload()
    matches = [data for data in uploaded.values() if hashlib.sha256(data).hexdigest() == ARCHIVE_SHA]
    if len(matches) != 1:
        raise ValueError('No se ha recibido el ZIP exacto de esta versión. Usa el ZIP que acompaña a este notebook.')
    archive_path.write_bytes(matches[0])
    del uploaded, matches

ROOT = Path(tempfile.mkdtemp(prefix='habitia-colab-', dir='/content'))
with zipfile.ZipFile(archive_path) as archive:
    for info in archive.infolist():
        p = PurePosixPath(info.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in info.filename:
            raise ValueError('Ruta no admitida en el ZIP.')
    archive.extractall(ROOT)
manifest = json.loads((ROOT / 'COLAB_MANIFIESTO.json').read_text())
for name, expected in manifest['sha256'].items():
    if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError('Archivo ausente o modificado: ' + name)
os.chdir(ROOT)
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
print('Código, datos y experimento: integridad correcta.')
print('Carpeta de trabajo:', ROOT)
print('Python:', sys.version.split()[0])


## B · Preparar las dependencias

In [ ]:
import subprocess, importlib, importlib.metadata

REQUIRED = {'numpy': '2.5.3', 'pandas': '3.0.5', 'scipy': '1.18.1', 'scikit-learn': '1.9.0', 'lightgbm': '4.7.0', 'pyarrow': '25.0.1', 'matplotlib': '3.11.1'}
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
missing = [f'{name}=={version}' for name, version in REQUIRED.items()
           if installed_version(name) != version]
if missing:
    print('Instalando las versiones del estudio. Espera a que termine esta celda…', flush=True)
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                             '--disable-pip-version-check', *missing],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (ROOT / 'instalacion-colab.log').write_text(result.stdout)
    if result.returncode:
        print(result.stdout[-6000:])
        raise RuntimeError('La instalación falló. Consulta el mensaje anterior.')
modules = {name: ('sklearn' if name == 'scikit-learn' else name) for name in REQUIRED}
loaded_different = [name for name, module in modules.items() if module in sys.modules
                    and getattr(sys.modules[module], '__version__', None) != REQUIRED[name]]
if loaded_different:
    raise RuntimeError('INSTALACIÓN COMPLETADA. Selecciona Entorno de ejecución → Reiniciar la sesión y ejecutar todas las celdas. No vuelvas a subir el ZIP.')
for name, module in modules.items():
    imported = importlib.import_module(module)
    if imported.__version__ != REQUIRED[name]:
        raise RuntimeError('Versión incompatible: ' + name)
print('Dependencias correctas. Continúa la comprobación del estudio.')


## C · Verificar el experimento

In [ ]:
OUT = ROOT / 'revision_2026-09-08' / 'experimento'
required_files = ['resultados_revision.json', 'predicciones_exteriores.parquet',
                  'predicciones_artefacto.parquet', 'particiones_exteriores.parquet',
                  'artefacto/manifiesto.json', 'artefacto/pipeline.json',
                  'artefacto/modelo_precio.txt', 'artefacto/modelo_q_lo.txt', 'artefacto/modelo_q_hi.txt']
missing = [name for name in required_files if not (OUT / name).is_file()]
if missing:
    raise FileNotFoundError('Falta evidencia del experimento. No se entrenará automáticamente: ' + ', '.join(missing))
subprocess.run([sys.executable, 'tests/verify_experiment_partitions.py'], check=True)
print('Preparación completada. Se reutiliza el experimento guardado, sin entrenar.')


# Estudio y resultados

Las celdas siguientes proceden del cuaderno canónico. Se conservan los métodos y cálculos, se adapta la ruta y se sustituye la celda de entrenamiento por una lectura obligatoria de la evidencia. El código Python del modelo se conserva íntegro.


In [ ]:
from pathlib import Path
import sys, os, json, inspect, hashlib
from datetime import datetime

ROOT = Path.cwd()
assert (ROOT / 'src' / 'train_revision.py').is_file(), 'Ejecuta antes las celdas de preparación.'
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('OMP_NUM_THREADS', '2')
import sklearn  # Antes de LightGBM para evitar el conflicto OpenMP observado en macOS.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code
from revision_data import load_revision_data, asset_weights, ORIGINAL_COLUMNS
from model_pipeline import ModelPipeline, historical_to_payloads
from train_revision import (CONFIG, prepare_data, run_experiment, metric_values,
                            select_parameters, calibrate_groups, assert_group_separation)

pd.set_option('display.max_colwidth', 100)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
REENTRENAR = False
OUT = ROOT / 'revision_2026-09-08' / 'experimento'
assert REENTRENAR is False, 'Este cuaderno comprueba la entrega; usa el cuaderno canónico para reentrenar.'
print('Python:', sys.version)
print('Resultados:', OUT)

## 1 La pregunta de investigación

¿Cuánto mejora un modelo que considera las características y la localización de la vivienda frente a la mediana de €/m² del barrio, cuando recibe los campos que puede proporcionar la aplicación?

La etiqueta `PRICE` es el precio anunciado. Primero estimamos un precio y su intervalo; después comparamos el anuncio con esa referencia. La segunda operación genera una **señal descriptiva**, no la predicción de una etiqueta independiente de ganga. Al entrenar no utilizamos ni `PRICE`, ni `UNITPRICE`, ni rentabilidades como variables explicativas.

## 2 Origen de los datos y qué significan

La fuente es [Idealista18](https://paezha.github.io/idealista18/), publicada por Rey-Blanco, Arbués, López y Páez (2024), con licencia [ODbL 1.0](https://paezha.github.io/idealista18/LICENSE.html). Sus precios tienen perturbación de aproximadamente ±2,5 % y redondeo; las coordenadas también fueron desplazadas. Por ello, una posición en el mapa o una diferencia pequeña de precio no debe interpretarse como una medición exacta.

El archivo local incluye enriquecimientos. Se detectaron 37 duplicaciones que coinciden en las 41 columnas originales y solo difieren en información de alquiler. Se consolidan por identidad original y se guarda qué filas se fusionaron. Las observaciones distintas de un mismo activo se conservan: no hay una fecha intratrimestral que permita elegir de forma fiable cuál sería la última.

In [ ]:
data, audit = prepare_data()
display(pd.DataFrame({'variable_original': ORIGINAL_COLUMNS}))
display(Markdown('### Registro de preparación'))
print(json.dumps(audit, ensure_ascii=False, indent=2)[:16000])
print('Observaciones elegibles:', len(data))
print('Inmuebles únicos:', data.ASSETID.nunique())
display(data[['PRICE', 'CONSTRUCTEDAREA', 'ROOMNUMBER', 'BATHNUMBER']].describe())

## 3 Limpieza y unidad de observación

Se aplican reglas fijas de validez y ámbito. No se calculan percentiles del precio sobre todo el conjunto para eliminar los extremos del test. Los valores extremos positivos que cumplen el dominio siguen participando, aunque hagan más difícil la predicción.

Cada fila es una observación; `ASSETID` agrupa observaciones del mismo inmueble. En cada partición, todas las filas de un activo permanecen juntas. Al ajustar, cada fila recibe un peso inverso al número de observaciones de su activo, normalizado para que la media del peso sea uno. Así, los activos más repetidos no dominan el ajuste.

Las métricas principales se calculan por observación para mantener una lectura comparable, y se añade una métrica por activo. Los intervalos de confianza descriptivos de las métricas remuestrean activos completos.

In [ ]:
sizes = data.groupby('ASSETID').size()
display(pd.Series({'activos': len(sizes), 'activos_con_varias_filas': int((sizes > 1).sum()),
                   'maximo_filas_por_activo': int(sizes.max()),
                   'activos_en_varios_trimestres': int((data.groupby('ASSETID').PERIOD.nunique() > 1).sum())}))
weights = asset_weights(data)
check_weights = pd.Series(np.asarray(weights)).groupby(data.ASSETID.to_numpy()).sum()
print('Peso total mínimo/máximo por activo:', check_weights.min(), check_weights.max())
assert np.allclose(check_weights.min(), check_weights.max())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(np.log10(data.PRICE), bins=55, color='#2F6F4E')
ax[0].set(xlabel='log10 del precio anunciado en euros', ylabel='Observaciones', title='Precio histórico sin recorte por percentiles')
data.PERIOD.value_counts().sort_index().plot.bar(ax=ax[1], color='#526F8A')
ax[1].set(xlabel='Trimestre de extracción', ylabel='Observaciones', title='Distribución temporal')
plt.tight_layout(); plt.show()

## 4 Un único contrato de entrada

El entrenamiento y el servicio llaman a `ModelPipeline`. `fit` aprende medianas, categorías y centroides solo del entrenamiento. `transform` usa esas estadísticas ya fijadas sobre cualquier anuncio posterior. Las coordenadas se comparan siempre con la misma distancia geodésica en kilómetros.

El principal prescinde de Catastro y alquiler: no se ha podido acreditar su disponibilidad en 2018. Utiliza atributos observables del anuncio, variables derivadas de ellos y barrio/distrito aproximados desde los centroides de entrenamiento. Los indicadores de dato ausente distinguen un valor desconocido de un `False` observado.

El histórico no conserva `propertyType`. El adaptador utiliza estudio → dúplex → piso como prioridad técnica, documentando ese supuesto. `ISINTOPFLOOR` significa última planta y **no se transforma en ático**. La comprobación espacial de 1 km al centroide aprendido describe soporte del modelo; no es un límite municipal ni una geocodificación exacta.

In [ ]:
# Ejemplo pedagógico: este fit se limita a 500 filas y NO es el modelo evaluado.
demo_rows = data.sample(500, random_state=17).reset_index(drop=True)
demo_pipeline = ModelPipeline().fit(demo_rows)
payloads = historical_to_payloads(demo_rows.head(3), include_price=True)
display(pd.DataFrame(payloads).drop(columns=['price']))
demo_x = demo_pipeline.transform(payloads, enforce_support=False)
display(demo_x.T)
forbidden = {'PRICE', 'UNITPRICE', 'rentabilidad_pct'}
assert not forbidden.intersection(demo_x.columns)
assert not any(c.startswith('CAD') or c.startswith('alq_') for c in demo_x.columns)
print('Número de variables del contrato:', len(demo_x.columns))
display(Code(inspect.getsource(ModelPipeline.transform), language='python'))

## 5 Selección evaluación y calibración

La evaluación exterior tiene tres folds por activo. Dentro de cada uno se reserva un 20 % de los grupos de desarrollo para calibración. El resto permite elegir parámetros con tres folds internos. Cada fold interno vuelve a aprender su propia preparación: no reutiliza medianas o centroides calculados con su validación.

Se comparan una referencia territorial, una regresión hedónica regularizada y LightGBM. La búsqueda de LightGBM considera seis configuraciones predefinidas; Ridge considera tres penalizaciones. Se elige mediante el MdAPE medio interno. No hay early stopping ni selección de variables a partir de la evaluación exterior.

Para el artefacto entregado se añade una partición histórica fija: 20 % de activos para evaluación y, del desarrollo restante, 20 % para calibración. Se conserva exactamente ese artefacto después de evaluarlo; no se reajusta silenciosamente con su test. El diagnóstico temporal ajusta y selecciona únicamente dentro de Q1–Q3 y evalúa Q4. Todas estas mediciones siguen siendo retrospectivas.

In [ ]:
display(pd.DataFrame(CONFIG['lightgbm_candidates']))
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'lightgbm_candidates'}, ensure_ascii=False, indent=2))
display(Code(inspect.getsource(select_parameters), language='python'))

## 6 Función objetivo y precio estimado

LightGBM minimiza el error absoluto de `log(PRICE)`. La transformación logarítmica reduce la escala de diferencias entre viviendas baratas y caras; al exponenciar se obtiene una estimación aproximadamente mediana del precio condicional. No se presenta como esperanza matemática exacta ni se añade una corrección de media aprendida del test.

La referencia territorial calcula una mediana ponderada de €/m² por barrio con respaldo por distrito y global, siempre desde entrenamiento. Ridge aporta una referencia lineal regularizada con transformaciones deterministas y efectos de zona. Todos reciben las mismas observaciones aceptadas por el contrato.

## 7 Intervalos y calibración por inmueble

Se entrenan modelos cuantílicos en los niveles 0,05 y 0,95. Antes de calibrar se ordenan los bordes y se amplían para incluir el punto. Para cada observación de calibración calculamos el máximo de los dos errores laterales en escala logarítmica. Para cada activo retenemos el peor de sus registros. Un estadístico de orden con corrección por muestra finita determina `Q`, que se limita inferiormente a cero.

Este diseño evita tratar las repeticiones del mismo activo como observaciones independientes de calibración. Se informa cobertura por fila y cobertura de activos cuyos registros quedan todos dentro. La cobertura observada no garantiza un 90 % por vivienda, por barrio ni en 2026. Véase [Romano, Patterson y Candès (2019)](https://arxiv.org/abs/1905.03222).

In [ ]:
display(Code(inspect.getsource(calibrate_groups), language='python'))

## 8 Cargar el experimento guardado

Esta copia comprueba la configuración y lee los resultados. Si falta el experimento, se detiene. Las celdas siguientes recalculan métricas y gráficos desde las predicciones guardadas.


In [ ]:
result_path = OUT / 'resultados_revision.json'
if not result_path.is_file():
    raise FileNotFoundError('Faltan resultados guardados; esta copia no inicia entrenamientos.')
results = json.loads(result_path.read_text())
assert results['configuracion'] == CONFIG, 'El resultado usa otra configuración.'
print('Se leen resultados completos de:', results['created_at'])
print('Duración del entrenamiento histórico, minutos:', round(results['duracion_segundos'] / 60, 2))
print('Hash de configuración:', results['config_sha256'])


## 9 Interpretar el error correctamente

MdAPE es la mediana del error porcentual absoluto. Un 10 % significa que la mitad de los errores porcentuales absolutos está por debajo de ese valor, no una exactitud del 90 %. El error absoluto en euros muestra qué significa la desviación monetaria. El sesgo firmado ayuda a detectar sobreestimación o infraestimación sistemática.

La tabla exterior combina predicciones hechas fuera del entrenamiento de cada activo. No debe confundirse con la tabla del artefacto final, que corresponde a un ajuste distinto y es la que describe el modelo exportado.

In [ ]:
outer = pd.read_parquet(OUT / 'predicciones_exteriores.parquet')
final = pd.read_parquet(OUT / 'predicciones_artefacto.parquet')
display(Markdown('### Evaluación exterior retrospectiva'))
display(pd.DataFrame(results['evaluacion_exterior']['metricas']).T.round(3))
display(Markdown('### Artefacto exportado evaluado en su reserva histórica'))
display(pd.DataFrame(results['artefacto_final']['metricas']).T.round(3))
print('Abstenciones exteriores:', results['evaluacion_exterior']['abstenciones'])
print('Elegibles exteriores:', results['evaluacion_exterior']['n_elegibles'])
display(pd.DataFrame(results['evaluacion_exterior']['bootstrap']))
assert outer.source_row_id.is_unique
recomputed = metric_values(outer.PRICE, outer.pred_lgb, outer.ASSETID,
                           outer.limite_inferior, outer.limite_superior)
assert np.isclose(recomputed['MdAPE_pct'], results['evaluacion_exterior']['metricas']['LightGBM']['MdAPE_pct'])

## 10 Comparación visual y diagnóstico de residuos

Los gráficos permiten ver si la mejora agregada oculta zonas de precio con errores grandes. La diagonal representa coincidencia entre precio anunciado y estimado; no una validación del valor de compraventa. El gráfico de residuos usa el signo: positivo indica sobreestimación.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
table = pd.DataFrame(results['evaluacion_exterior']['metricas']).T
axes[0].barh(table.index, table.MdAPE_pct, color=['#9BA5AA', '#526F8A', '#2F6F4E'])
axes[0].set(xlabel='MdAPE (%)', title='Comparación con las mismas filas')
s = outer.sample(min(4000, len(outer)), random_state=17)
axes[1].scatter(s.PRICE, s.pred_lgb, s=4, alpha=.2, color='#2F6F4E')
limits = [outer.PRICE.min(), outer.PRICE.max()]
axes[1].plot(limits, limits, '--', color='black', linewidth=1)
axes[1].set(xscale='log', yscale='log', xlabel='Precio anunciado (€ de 2018)', ylabel='Estimación (€ de 2018)', title='Predicción exterior')
axes[2].scatter(s.PRICE, (s.pred_lgb-s.PRICE)/s.PRICE*100, s=4, alpha=.2, color='#526F8A')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set(xscale='log', xlabel='Precio anunciado (€ de 2018)', ylabel='Error firmado (%)', title='Sesgo y dispersión')
plt.tight_layout(); plt.show()

## 11 Errores por distrito y por tramo de precio

Se publican tamaños de muestra junto a las métricas. Los deciles son un diagnóstico calculado después de evaluar: no eliminan observaciones ni deciden parámetros. Una cobertura agregada cercana al nivel nominal puede ocultar un comportamiento distinto en viviendas baratas o caras. El remuestreo por activo de la tabla anterior estima variación descriptiva de los casos evaluados; no incluye toda la incertidumbre de volver a entrenar el sistema.

<!-- HABITIA_INTERPRETACION_JSON -->
**Lectura de la ejecución entregada.** En el decil más barato, con 9.498 anuncios de 21.000 a 118.000 € de 2018, el MdAPE es **16,24 %**, el sesgo mediano es **+14,86 %** (sobreestimación) y la cobertura por anuncio es **81,81 %**. Eso deja aproximadamente 18,19 de cada 100 anuncios de este tramo fuera de su intervalo. La cobertura global oculta aquí una limitación real: una brecha negativa en una vivienda barata necesita revisión, pues puede reflejar un error sistemático del modelo. Estos deciles usan el precio real anunciado para diagnosticar después de evaluar; no constituyen una garantía ni una regla de recalibración validada para anuncios nuevos.

*Cifras tomadas de `revision_2026-09-08/experimento/resultados_revision.json`, ejecución registrada 2026-09-08T10:56:07.779466+00:00. Si se ejecuta otro experimento, se deben renovar estas interpretaciones a partir de su JSON.*

In [ ]:
segments = results['evaluacion_exterior']['segmentos']
display(pd.DataFrame(segments['distrito'])[['segmento','n','n_activos','MdAPE_pct','sesgo_mediano_pct','cobertura_pct','anchura_relativa_mediana_pct']].round(2))
deciles = pd.DataFrame(segments['decil_precio'])
display(deciles.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(deciles.segmento, deciles.MdAPE_pct, 'o-', color='#2F6F4E')
axes[0].set(xlabel='Decil de precio anunciado', ylabel='MdAPE (%)', title='Error por precio')
axes[1].plot(deciles.segmento, deciles.cobertura_pct, 'o-', color='#526F8A')
axes[1].axhline(90, color='gray', linestyle='--', label='Nivel nominal 90 %')
axes[1].set(xlabel='Decil de precio anunciado', ylabel='Cobertura observada (%)', title='Cobertura por precio')
axes[1].legend(); plt.tight_layout(); plt.show()

## 12 Diagnóstico temporal

Los parámetros del diagnóstico temporal se seleccionan únicamente con Q1–Q3. Q4 nunca se utiliza en ese ajuste o calibración. Aun así, el análisis sigue siendo retrospectivo porque el proyecto ya había inspeccionado Q4 antes de esta revisión. La diferencia entre resultados agrupados y temporales ayuda a discutir estabilidad; no valida por sí sola el salto hasta 2026.

<!-- HABITIA_INTERPRETACION_JSON -->
**Lectura de la ejecución entregada.** En los 43.839 anuncios aceptados de Q4, el MdAPE es **11,22 %**, frente al 10,31 % de la evaluación exterior agrupada. El sesgo mediano de **-4,24 %** indica infraestimación: la estimación tiende a quedar por debajo del precio anunciado. La cobertura es **89,35 % por observación** y **87,88 % por activo con todos sus registros cubiertos**; la unidad de medida importa. El modelo cambia de comportamiento al evaluarse en otro período, pero esta diferencia no se puede atribuir solo al calendario: también cambian el tamaño de ajuste y la composición de la muestra. No ofrece evidencia de precisión en 2026.

*Cifras tomadas de `revision_2026-09-08/experimento/resultados_revision.json`, ejecución registrada 2026-09-08T10:56:07.779466+00:00. Si se ejecuta otro experimento, se deben renovar estas interpretaciones a partir de su JSON.*

In [ ]:
if results['temporal'] is not None:
    display(pd.DataFrame(results['temporal']['metricas']).T.round(3))
    print('Abstenciones temporales:', results['temporal']['abstenciones_evaluacion'])
    display(results['temporal']['particiones'])

## 13 Interpretabilidad del modelo evaluado

Se calculan contribuciones SHAP con LightGBM sobre una muestra fijada de observaciones exteriores. La magnitud media absoluta indica cuánto suele contribuir cada variable a la salida del modelo en log euros. No significa cuánto aumentaría causalmente el precio si reformáramos una vivienda. Variables correlacionadas pueden repartirse la atribución de maneras distintas.

In [ ]:
shap_parts = [pd.Series(fold['shap']['media_absoluta']) for fold in results['evaluacion_exterior']['folds']]
shap_mean = pd.concat(shap_parts, axis=1).mean(axis=1).sort_values()
display(shap_mean.sort_values(ascending=False).to_frame('SHAP absoluto medio en log euros'))
fig, ax = plt.subplots(figsize=(9, 5))
shap_mean.tail(15).plot.barh(ax=ax, color='#2F6F4E')
ax.set(xlabel='Contribución absoluta media en log euros', title='Variables del modelo evaluado')
plt.tight_layout(); plt.show()

## 14 Comprobar el artefacto servido

La prueba siguiente carga los archivos exportados a través de `Valorador`, reconstruye anuncios y contrasta sus respuestas con las predicciones registradas. Cambia después únicamente el precio anunciado: la estimación debe permanecer idéntica, aunque la brecha cambie. No se llama a Fly ni a Idealista.

También se comprueba que el artefacto y sus metadatos tienen los hashes declarados. Una carpeta que mezcla modelos y preparación de ejecuciones distintas no es un despliegue válido.

In [ ]:
from valorador import Valorador
artifact = OUT / 'artefacto'
manifest = json.loads((artifact / 'manifiesto.json').read_text())
for name, expected in manifest['sha256'].items():
    assert hashlib.sha256((artifact / name).read_bytes()).hexdigest() == expected
v = Valorador(artifact)
chosen = final.head(20)
rows = data.set_index('source_row_id').loc[chosen.source_row_id].reset_index()
ads = historical_to_payloads(rows, include_price=True)
responses = v.valorar(ads, renivelar=False)
served = np.asarray([r['precio_estimado'] for r in responses])
assert np.max(np.abs(served - chosen.pred_lgb.to_numpy())) <= 0.500001
assert all(r['intervalo'][0] <= r['precio_estimado'] <= r['intervalo'][1] for r in responses)
changed = {**ads[0], 'price': ads[0]['price'] * 1.5}
assert v.valorar(changed)['precio_estimado'] == responses[0]['precio_estimado']
display(pd.DataFrame(responses)[['precio_anunciado','precio_estimado','intervalo','brecha_pct','barrio','nivel_precios']])
print('Paridad con el servicio verificada en', len(responses), 'casos; diferencia máxima permitida: redondeo a euros.')

## 15 Cómo leer una señal de precio

Un anuncio situado por debajo del borde inferior merece revisión frente al modelo, pero puede tener información omitida, defectos o condiciones que expliquen su precio. La etiqueta no se ha validado frente a operaciones cerradas ni a valoraciones independientes de especialistas. Una rentabilidad definida como alquiler dividido por el mismo precio de venta no sirve como prueba independiente de ganga.

El servicio diferencia el nivel histórico de 2018 del escenario indexado mediante el factor heredado del IPV. Aplicar un índice agregado no verifica el comportamiento de cada barrio o inmueble. Las respuestas indexadas deben mostrar ese carácter exploratorio.

In [ ]:
historical = v.valorar(ads[0], renivelar=False)
scenario = v.valorar(ads[0], renivelar=True)
display(pd.DataFrame([{'escenario': '2018', 'estimacion': historical['precio_estimado'],
                       'intervalo': historical['intervalo']},
                      {'escenario': scenario['nivel_precios'] + ' indexado',
                       'estimacion': scenario['precio_estimado'], 'intervalo': scenario['intervalo']}]))
print('Factor de escenario:', manifest['renivelado']['factor'])
print('Precisión actual validada:', scenario['precision_actual_validada'])
try:
    v.valorar({**ads[0], 'municipality': 'Barcelona'})
except ValueError as e:
    print('Abstención esperada:', e)

## 16 Conclusiones y límites que deben aparecer en la defensa

La conclusión se obtiene comparando la tabla real con la referencia territorial, revisando sesgo, cobertura, anchura y abstenciones. Una mejora del promedio no elimina los fallos en segmentos concretos. El modelo exportado recibe la misma preparación que el evaluado y conserva por separado sus grupos de calibración y evaluación.

Persisten límites sustantivos: datos históricos ya explorados, precios de oferta perturbados, coordenadas aproximadas, tipo bruto de inmueble ausente, falta de muestra contemporánea y ausencia de etiquetas independientes de oportunidad. Un resultado honesto puede ser menos espectacular que el de una evaluación contaminada y, aun así, aportar más evidencia.

### Archivos para reproducir y auditar

- `src/revision_data.py`: origen, consolidación, exclusiones y pesos.
- `src/model_pipeline.py`: preparación compartida.
- `src/train_revision.py`: selección, evaluación, calibración y exportación.
- `src/valorador.py` y `servicio/api.py`: inferencia y contrato HTTP.
- `revision_2026-09-08/experimento`: configuración, particiones, predicciones, métricas y artefacto.
- `docs/verificacion-entrega.md`: comprobaciones de la distribución y límites de la entrega.

### Referencias metodológicas

- [Rey-Blanco et al. (2024), datos de Idealista](https://journals.sagepub.com/doi/10.1177/23998083241242844).
- [scikit-learn, preparación y prevención de fugas](https://scikit-learn.org/stable/common_pitfalls.html).
- [scikit-learn, evaluación con grupos](https://scikit-learn.org/stable/modules/cross_validation.html).
- [Romano, Patterson y Candès (2019), Conformalized Quantile Regression](https://arxiv.org/abs/1905.03222).
- [Lundberg y Lee (2017), A Unified Approach to Interpreting Model Predictions](https://arxiv.org/abs/1705.07874).
- [LightGBM, Booster.predict y contribuciones nativas pred_contrib](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.Booster.html).

## Prueba final · todas las predicciones del artefacto

In [ ]:
subprocess.run([sys.executable, 'tests/verify_exported_evaluation.py'], check=True)
print('PRUEBA COMPLETA: métricas, gráficos, particiones y 18.782 predicciones verificadas. No se ha reentrenado el modelo.')
